# Figure 1 — what $t_0$ is

Defines the $t_0$ speed metric on the cleanest example: CIFAR-10 with **random labels**, trained
**from scratch**. Dots are measured train accuracy from `metrics.json`; the line is the fixed
floor/ceiling logistic $y = b + l\,\sigma((t-t_0)/s)$ ($b=0.1$ chance, $l=0.9$), fitted by
`common.fit_t0`; red marks $t_0$, its inflection.

- **Consumes:** `checkpoints/upstream_randlabel_seed21_labelseed21/metrics.json`
  — produced by `python train/train.py --stage upstream --task randlabel --seed 21 --label-seed 21`
- **Produces:** `outputs/fig1_t0_definition.png`
- Runtime: seconds (one fit + one plot).


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# put the repo root (the dir that holds common/) on sys.path, from any working directory
_root = Path.cwd()
while not (_root / "common").is_dir() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

from common.config import FIGURE_DATA_DIR, OUTPUT_DIR
from common.fit_t0 import fit_t0

RUN = FIGURE_DATA_DIR / "upstream_randlabel_seed21_labelseed21"
OUT = OUTPUT_DIR / "fig1_t0_definition.png"

In [ ]:
d = json.loads((RUN / "metrics.json").read_text(encoding="utf-8"))
assert d["config"].get("stage") == "upstream", "t0 is defined on a from-scratch (upstream) run"
epochs = np.array([r["epoch"] for r in d["epochs"]], dtype=np.float64)
train_acc = np.array([r["train_acc"] for r in d["epochs"]], dtype=np.float64)

fit = fit_t0(epochs, train_acc)
resid = fit.residuals(epochs, train_acc)
print(f"seed={d['config']['seed']} label_seed={d['config']['label_seed']} epochs={len(epochs)}")
print(f"t0={fit.t0:.2f}  s={fit.s:.2f}  R^2={fit.r2:.4f}  mse={fit.mse:.2e}")
print(f"max|residual|={np.abs(resid).max():.3f} at epoch {int(epochs[np.abs(resid).argmax()])}")

In [ ]:
W_IN, H_IN, DPI = 7.0, 4.3, 400
PCT = 100.0  # fit is in fraction space (the t0 convention); percent lives only in the plot
C_DATA, C_FIT, C_T0 = plt.cm.viridis(0.5), "#1a1a1a", "#c0392b"
mpl.rcParams.update({"font.size": 9, "axes.linewidth": 0.9})

fig, ax = plt.subplots(figsize=(W_IN, H_IN))
t_dense = np.linspace(epochs.min(), epochs.max(), 800)
ax.plot(t_dense, fit.curve(t_dense) * PCT, "-", color=C_FIT, lw=1.6, alpha=0.9, label="fitted sigmoid")
ax.plot(epochs, train_acc * PCT, "o", ms=4.5, color=C_DATA, mec="white", mew=0.5, ls="none",
        label="train accuracy")
ax.axvline(fit.t0, color=C_T0, ls="--", lw=1.2, alpha=0.85)
ax.plot([fit.t0], [fit.midpoint_acc * PCT], "o", ms=7, color=C_T0, mec="white", mew=1.0)
ax.annotate(rf"$t_0 = {fit.t0:.1f}$", xy=(fit.t0, fit.midpoint_acc * PCT), xytext=(6, -16),
            textcoords="offset points", color=C_T0, fontsize=9)
ax.set_xlabel("Pre-training epoch")
ax.set_ylabel("Training accuracy (%)")
ax.set_xlim(0, epochs.max() + 1)
ax.set_ylim(0.0, 105.0)
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(fontsize=8, frameon=False, loc="upper left")
fig.tight_layout()
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=DPI)
print(f"saved {OUT}")